# Neural Network

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings("ignore")

In [4]:
stock_data = pd.read_csv('clusteredFinal22.csv')
employee_data = pd.read_csv('employee_risk_data_with_outliers.csv')

In [5]:
label_encoder = LabelEncoder()
employee_data['Risk_encoded'] = label_encoder.fit_transform(employee_data['Risk'])

In [6]:
employee_data.head()

,Age,Salary,Risk,Risk_encoded
0,56,77605,high,0
1,46,135616,medium,2
2,32,68088,medium,2
3,60,108752,high,0
4,25,85284,low,1


In [7]:
features = ['Age', 'Salary', 'Risk_encoded']

def map_risk_to_cluster(risk):
    if risk == 'low':
        return 2 if np.random.rand() < 0.5 else 3
    elif risk == 'medium':
        return 4
    elif risk == 'high':
        return 1 if np.random.rand() < 0.5 else 0
np.random.seed(0) 
employee_data['Cluster'] = employee_data['Risk'].apply(map_risk_to_cluster)
employee_data.head()

,Age,Salary,Risk,Risk_encoded,Cluster
0,56,77605,high,0,0
1,46,135616,medium,2,4
2,32,68088,medium,2,4
3,60,108752,high,0,0
4,25,85284,low,1,3


In [8]:
X = employee_data[features]
y = employee_data['Cluster']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [9]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [10]:
!pip install tensorflow;

Defaulting to user installation because normal site-packages is not writeable


In [10]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
model = Sequential()
model.add(Dense(32, input_dim=len(features), activation='relu'))
model.add(Dense(16, activation='relu'))
model.add(Dense(5, activation='softmax'))

In [11]:
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam')
model.fit(X_train_scaled, y_train, epochs=50, batch_size=8, validation_split=0.2)

Epoch 1/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - loss: 1.4514 - val_loss: 1.1762
Epoch 2/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 1.0629 - val_loss: 0.9217
Epoch 3/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.8779 - val_loss: 0.7489
Epoch 4/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.6682 - val_loss: 0.6600
Epoch 5/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.6172 - val_loss: 0.5971
Epoch 6/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.5848 - val_loss: 0.5597
Epoch 7/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.5470 - val_loss: 0.5390
Epoch 8/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.5130 - val_loss: 0.5289
Epoch 9/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.4910 - val_loss: 0.5225
Epoch 10/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.4860 - val_loss: 0.5089
Epoch 11/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.4820 - val_loss: 0.5107
Epoch 12/50
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.4978 - val_l

In [12]:
def predict_cluster(age, salary, risk, model, scaler, label_encoder):
    risk_encoded = label_encoder.transform([risk])[0]
    input_data = pd.DataFrame([[age, salary, risk_encoded]], columns=['Age', 'Salary', 'Risk_encoded'])
    input_data_scaled = scaler.transform(input_data)
    cluster = model.predict(input_data_scaled)
    return cluster.argmax()

In [13]:
import joblib

model.save('employee_risk_model.h5')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(label_encoder, 'label_encoder.pkl')


['label_encoder.pkl']

In [13]:
age = 30
salary = 80000
risk = 'low'

predicted_cluster=predict_cluster(age, salary, risk, model, scaler, label_encoder)
allocated_stocks = stock_data[stock_data['Cluster'] == predicted_cluster]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 166ms/step


In [14]:
allocated_stocks.head()

,Symbol,Company Name,Market cap,Beta,P/E,Price,Volume,Annual Return,Cluster
0,NESTLEIND.NS,Nestlé India Limited,2419889446450,0.042,79.86,2509.85,2028038,0.3844,3
110,AXISBANK.NS,Axis Bank Limited,3658873120000,0.882,14.04,1184.50,21038016,0.2261,3
139,BEL.NS,Bharat Electronics Limited,1903101223000,0.370,51.23,260.35,193451456,1.1655,3
204,TATAMTRDVR.NS,Tata Motors Limited,3506654985096,1.818,7.69,620.75,1879679,2.2015,3
231,TRENT.NS,Trent Limited,1743237150600,0.356,116.62,4903.80,1231937,1.9575,3


In [15]:
age = 40
salary = 200000
risk = 'low'

predicted_cluster = predict_cluster(age, salary, risk, model, scaler, label_encoder)
allocated_stocks = stock_data[stock_data['Cluster'] == predicted_cluster]
allocated_stocks.head()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step


,Symbol,Company Name,Market cap,Beta,P/E,Price,Volume,Annual Return,Cluster
383,RELIANCE.NS,Reliance Industries Limited,19224708135000,0.599,28.04,2841.50,8730609,0.3065,2
418,TCS.NS,Tata Consultancy Services Limited,13554993280500,0.519,29.83,3746.45,2799454,0.2503,2
700,HDFCBANK.NS,HDFC Bank Limited,11800151006000,0.768,17.58,1551.80,40806442,-0.0803,2


In [16]:
age = 42
salary = 15000
risk = 'medium'

predicted_cluster = predict_cluster(age, salary, risk, model, scaler, label_encoder)
allocated_stocks = stock_data[stock_data['Cluster'] == predicted_cluster]
allocated_stocks.head()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step


,Symbol,Company Name,Market cap,Beta,P/E,Price,Volume,Annual Return,Cluster
17,ZOMATO.NS,Zomato Limited,1622400762000,0.221,462.88,183.80,57186414,2.5532,4
286,URJA.NS,Urja Global Limited,11740887221,1.156,533.75,20.35,7046283,1.9531,4
299,ORICONENT.NS,Oricon Enterprises Limited,5300370000,0.738,1185.00,33.75,194781,1.2508,4
345,CUPID.NS,Cupid Limited,24004354750,0.497,626.00,89.45,514712,9.0463,4
384,POLICYBZR.NS,PB Fintech Limited,583563994991,0.649,868.19,1293.35,1637871,0.8046,4


In [17]:
age = 30
salary = 70000
risk = 'medium'
target_savings = 600000

predicted_cluster = predict_cluster(age, salary, risk, model, scaler, label_encoder)
allocated_stocks = stock_data[stock_data['Cluster'] == predicted_cluster]
allocated_stocks.head()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 111ms/step


,Symbol,Company Name,Market cap,Beta,P/E,Price,Volume,Annual Return,Cluster
17,ZOMATO.NS,Zomato Limited,1622400762000,0.221,462.88,183.80,57186414,2.5532,4
286,URJA.NS,Urja Global Limited,11740887221,1.156,533.75,20.35,7046283,1.9531,4
299,ORICONENT.NS,Oricon Enterprises Limited,5300370000,0.738,1185.00,33.75,194781,1.2508,4
345,CUPID.NS,Cupid Limited,24004354750,0.497,626.00,89.45,514712,9.0463,4
384,POLICYBZR.NS,PB Fintech Limited,583563994991,0.649,868.19,1293.35,1637871,0.8046,4


# Percentage Allocation

In [18]:
def filter_stocks(allocated_stocks, risk_tolerance):
    beta_min = allocated_stocks['Beta'].min()
    beta_max = allocated_stocks['Beta'].max()

    if risk_tolerance == 'low':
        threshold = beta_min + (beta_max - beta_min) / 3
        return allocated_stocks[allocated_stocks['Beta'] <= threshold]
    elif risk_tolerance == 'medium':
        lower_threshold = beta_min + (beta_max - beta_min) / 3
        upper_threshold = beta_min + 2 * (beta_max - beta_min) / 3
        return allocated_stocks[(allocated_stocks['Beta'] > lower_threshold) & (allocated_stocks['Beta'] <= upper_threshold)]
    else:  # high risk
        threshold = beta_min + 2 * (beta_max - beta_min) / 3
        return allocated_stocks[allocated_stocks['Beta'] > threshold]

def rank_stocks(allocated_stocks, target_savings):
    if target_savings > 1000000:
        allocated_stocks['Rank'] = allocated_stocks['Annual Return'].rank(ascending=False)
    else:
        allocated_stocks['Rank'] = allocated_stocks['Beta'].rank(ascending=True)
    return allocated_stocks.sort_values(by='Rank')

def allocate_percentage(allocated_stocks, num_stocks):
    selected_stocks = allocated_stocks.head(num_stocks).copy()
    total_rank = selected_stocks['Rank'].sum()
    
    selected_stocks['Allocation'] = selected_stocks['Rank'].apply(lambda rank: (total_rank - rank + 1) / total_rank)
    selected_stocks['Allocation'] = selected_stocks['Allocation'] / selected_stocks['Allocation'].sum() * 100
    return selected_stocks

def determine_number_of_stocks(salary, target_savings):
    if salary < 40000:
        return 2
    elif 40000 <= salary < 80000:
        return 3 + (1 if target_savings > 500000 else 0)
    else:
        return 5 + (1 if target_savings > 500000 else 0)

def generate_stocks(salary, age, risk_tolerance, target_savings, allocated_stocks):
    risk_tolerance = risk_tolerance.lower()
    
    filtered_data = filter_stocks(allocated_stocks, risk_tolerance)
    ranked_stocks = rank_stocks(filtered_data, target_savings)
    
    num_stocks = determine_number_of_stocks(salary, target_savings)
    
    finalStocks = allocate_percentage(ranked_stocks, num_stocks)
    return finalStocks

In [24]:
finalStocks = generate_stocks(salary, age, risk, target_savings, allocated_stocks)
cluster_value = finalStocks['Cluster'].iloc[0]
stock_data.head()

,Symbol,Company Name,Market cap,Beta,P/E,Price,Volume,Annual Return,Cluster
0,NESTLEIND.NS,Nestlé India Limited,2419889446450,0.042,79.86,2509.85,2028038,0.3844,3
1,IDFCFIRSTB.NS,IDFC First Bank Limited,546719880000,0.903,18.37,77.25,97310059,0.3722,1
2,KARURVYSYA.NS,The Karur Vysya Bank Limited,155266778900,0.859,9.92,194.05,3173706,0.8161,1
3,IBREALEST.NS,Indiabulls Real Estate Limited,79886813692,1.165,-6.92,126.80,13184531,1.3957,1
4,GMRP&UI.NS,GMR Power And Urban Infra Limited,38780913472,0.571,-9.62,64.25,6149127,1.6288,0


In [25]:
finalStocks.to_csv('FinalAllocatedStocks.csv', index=False)